# Parker (1958): Dynamics of the Interplanetary Gas and Magnetic Fields
# Parker (1958): 행성간 가스와 자기장의 역학

## Implementation of Key Concepts / 핵심 개념 구현

This notebook implements and visualizes the key theoretical results from Parker's 1958 paper,
which established the theoretical foundation of the solar wind and derived the Parker spiral
structure of the interplanetary magnetic field.

이 노트북은 Parker의 1958년 논문의 핵심 이론적 결과를 구현하고 시각화합니다.
이 논문은 태양풍의 이론적 기초를 확립하고 행성간 자기장의 Parker 나선 구조를 유도했습니다.

**Contents / 목차:**
1. Hydrostatic Equilibrium Failure / 정역학 평형의 실패
2. Parker Equation and Trans-sonic Solutions / Parker 방정식과 Trans-sonic 해
3. Reproducing Parker's Fig. 1 — Velocity Profiles / Fig. 1 재현 — 속도 분포
4. Critical Point Analysis / 임계점 분석
5. Parker Spiral — Interplanetary Magnetic Field / Parker 나선 — 행성간 자기장
6. Mass Loss Rate and Energy Budget / 질량 손실률과 에너지 예산
7. Comparison with Modern Observations / 현대 관측과의 비교

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import solve_ivp

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'legend.fontsize': 11,
    'figure.dpi': 100,
})

# Physical constants in CGS
k_B = 1.381e-16         # Boltzmann constant [erg/K]
G_grav = 6.674e-8       # Gravitational constant [cm³/g/s²]
M_sun = 1.989e33        # Solar mass [g]
R_sun = 6.957e10        # Solar radius [cm]
M_H = 1.673e-24         # Hydrogen atom mass [g]
AU = 1.496e13           # 1 AU [cm]
omega_sun = 2.7e-6      # Solar angular velocity [rad/s]

# Parker's model parameters
a = 1e11                # Coronal base radius [cm] = 10⁶ km ≈ 1.44 R_sun
N_0 = 3e7               # Base density [cm⁻³]

print("Constants loaded / 상수 로드 완료")
print(f"  a = {a:.0e} cm = {a/R_sun:.2f} R_sun")
print(f"  1 AU / a = {AU/a:.1f}")

## Part 1: Hydrostatic Equilibrium Failure
## 파트 1: 정역학 평형의 실패

Parker's key argument: if the corona is in hydrostatic equilibrium with temperature
falling as $T(r) = T_0 (a/r)^{1/(n+1)}$ due to thermal conduction, the pressure at infinity
is **finite** and much larger than interstellar pressure. Therefore, equilibrium is impossible.

Parker의 핵심 논증: 코로나가 열전도에 의해 $T(r) = T_0 (a/r)^{1/(n+1)}$로 감소하는 온도에서
정역학 평형을 이룬다면, 무한대에서의 압력이 **유한**하고 성간 압력보다 훨씬 크다.
따라서 평형은 불가능하다.

In [ ]:
def compute_lambda(T0, a_cm=a):
    """Compute Parker's dimensionless parameter lambda = GM_sun * M_H / (2kT0 * a).

    Args:
        T0: Coronal base temperature [K].
        a_cm: Coronal base radius [cm].

    Returns:
        Dimensionless lambda.
    """
    return G_grav * M_sun * M_H / (2 * k_B * T0 * a_cm)


def hydrostatic_pressure(r_over_a, T0, n_cond, N0=N_0):
    """Compute hydrostatic pressure profile (Parker Eq. 7).

    Args:
        r_over_a: Array of r/a values.
        T0: Base temperature [K].
        n_cond: Thermal conductivity exponent (5/2 for ionized H, 1/2 for neutral).
        N0: Base density [cm⁻³].

    Returns:
        Pressure [dyne/cm²].
    """
    lam = compute_lambda(T0)
    p0 = 2 * N0 * k_B * T0
    exponent = (lam * (n_cond + 1) / n_cond) * ((1.0 / r_over_a)**(n_cond / (n_cond + 1)) - 1)
    return p0 * np.exp(exponent)


# Temperature profiles
r_over_a = np.logspace(0, 4, 500)  # r/a from 1 to 10⁴

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

T0_ref = 1.5e6  # Reference temperature

# Left: Temperature profile
ax1 = axes[0]
for n, label, color in [(2.5, 'Ionized H (n=5/2)', 'red'), (0.5, 'Neutral H (n=1/2)', 'blue')]:
    T_profile = T0_ref * (1.0 / r_over_a)**(1.0 / (n + 1))
    ax1.loglog(r_over_a, T_profile, color=color, linewidth=2, label=label)

ax1.axhline(y=100, color='green', linestyle='--', alpha=0.7, label='ISM: 100 K')
ax1.axvline(x=AU/a, color='gray', linestyle=':', alpha=0.5, label=f'1 AU (ξ={AU/a:.0f})')
ax1.set_xlabel('ξ = r/a')
ax1.set_ylabel('T(r) [K]')
ax1.set_title('Temperature Profile\n온도 분포')
ax1.legend(fontsize=9)
ax1.set_ylim(10, 3e6)
ax1.grid(True, alpha=0.3)

# Middle: Pressure profile
ax2 = axes[1]
p_ISM = 10 * k_B * 100 * 2  # 10 atoms/cm³, 100 K
for n, label, color in [(2.5, 'Ionized H', 'red'), (0.5, 'Neutral H', 'blue')]:
    p = hydrostatic_pressure(r_over_a, T0_ref, n)
    ax2.loglog(r_over_a, p, color=color, linewidth=2, label=label)

ax2.axhline(y=p_ISM, color='green', linewidth=2, linestyle='--', label=f'p_ISM = {p_ISM:.1e} dyne/cm²')
ax2.axvline(x=AU/a, color='gray', linestyle=':', alpha=0.5)
ax2.set_xlabel('ξ = r/a')
ax2.set_ylabel('p(r) [dyne/cm²]')
ax2.set_title('Pressure: Hydrostatic Equilibrium\n압력: 정역학 평형')
ax2.legend(fontsize=9)
ax2.set_ylim(1e-10, 1)
ax2.grid(True, alpha=0.3)

# Right: p(∞) vs T0
ax3 = axes[2]
T0_range = np.linspace(0.5e6, 4e6, 100)
p0_range = 2 * N_0 * k_B * T0_range

for n, label, color in [(2.5, 'Ionized H (n=5/2)', 'red'), (0.5, 'Neutral H (n=1/2)', 'blue')]:
    lam_range = compute_lambda(T0_range)
    p_inf = p0_range * np.exp(-lam_range * (n + 1) / n)
    ax3.semilogy(T0_range / 1e6, p_inf, color=color, linewidth=2, label=label)

ax3.axhline(y=p_ISM, color='green', linewidth=2, linestyle='--', label=f'p_ISM')
ax3.set_xlabel('T₀ [10⁶ K]')
ax3.set_ylabel('p(∞) [dyne/cm²]')
ax3.set_title('Pressure at Infinity vs T₀\n무한대 압력 vs 코로나 온도')
ax3.legend(fontsize=9)
ax3.set_ylim(1e-15, 1e-1)
ax3.grid(True, alpha=0.3)

# Annotate the key conclusion
ax3.annotate('p(∞) ≫ p_ISM\n→ Equilibrium\n   IMPOSSIBLE!\n→ 평형 불가능!',
             xy=(1.5, 7e-6), fontsize=11, fontweight='bold', color='red',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

plt.tight_layout()
plt.savefig('part1_hydrostatic_failure.png', dpi=150, bbox_inches='tight')
plt.show()

# Print key numbers
lam = compute_lambda(T0_ref)
p0 = 2 * N_0 * k_B * T0_ref
print(f"\nParker's key numbers (T₀ = {T0_ref/1e6:.1f} × 10⁶ K):")
print(f"  λ = GM☉M/(2kT₀a) = {lam:.2f}")
print(f"  p₀ = 2N₀kT₀ = {p0:.2e} dyne/cm²")
print(f"  p(∞) ionized (n=5/2) = {p0 * np.exp(-lam * 3.5/2.5):.2e} dyne/cm²")
print(f"  p(∞) neutral (n=1/2) = {p0 * np.exp(-lam * 1.5/0.5):.2e} dyne/cm²")
print(f"  p_ISM = {p_ISM:.2e} dyne/cm²")
print(f"  Ratio p(∞)/p_ISM [ionized] = {p0 * np.exp(-lam * 3.5/2.5) / p_ISM:.0e}")
print(f"  → Hydrostatic equilibrium is IMPOSSIBLE! / 정역학 평형 불가능!")

## Part 2: Parker Equation — Isothermal Trans-sonic Solutions
## 파트 2: Parker 방정식 — 등온 Trans-sonic 해

The Parker equation (Eq. 14) for isothermal expansion:

$$\psi - \ln\psi = \psi_0 - \ln\psi_0 + 4\ln\xi - 2\lambda\left(1 - \frac{1}{\xi}\right)$$

where $\psi = Mv^2/2kT_0$ and $\xi = r/a$. The trans-sonic solution requires $\psi = 1$ at $\xi = \lambda/2$.

Parker 방정식 (Eq. 14) — 등온 팽창:
trans-sonic 해는 임계점 $\xi = \lambda/2$에서 $\psi = 1$ (즉, $v = c_s$)을 요구합니다.

In [ ]:
def parker_equation_rhs(xi, lam):
    """Compute the right-hand side function of Parker's isothermal equation.

    Parker Eq. 14: psi - ln(psi) = psi_0 - ln(psi_0) + 4*ln(xi) - 2*lambda*(1 - 1/xi)
    Define F(xi) = 4*ln(xi) - 2*lambda*(1 - 1/xi)
    Then: psi - ln(psi) = C + F(xi) where C = psi_0 - ln(psi_0) - F(1)

    Args:
        xi: Dimensionless distance r/a.
        lam: Dimensionless parameter lambda.

    Returns:
        F(xi) value.
    """
    return 4.0 * np.log(xi) - 2.0 * lam * (1.0 - 1.0/xi)


def solve_psi_from_Y(Y_val):
    """Solve psi - ln(psi) = Y_val for psi.

    This equation has two solutions: one < 1 (subsonic) and one > 1 (supersonic).
    For the trans-sonic solution, we use the supersonic branch for xi > xi_c
    and subsonic branch for xi < xi_c.

    Args:
        Y_val: Value of psi - ln(psi).

    Returns:
        Tuple of (psi_subsonic, psi_supersonic).
    """
    # psi - ln(psi) = Y  has minimum Y=1 at psi=1
    if Y_val < 1.0:
        return np.nan, np.nan

    # Subsonic branch: 0 < psi < 1
    try:
        psi_sub = brentq(lambda p: p - np.log(p) - Y_val, 1e-10, 1.0)
    except ValueError:
        psi_sub = np.nan

    # Supersonic branch: psi > 1
    try:
        psi_sup = brentq(lambda p: p - np.log(p) - Y_val, 1.0, 100.0)
    except ValueError:
        try:
            psi_sup = brentq(lambda p: p - np.log(p) - Y_val, 1.0, 1000.0)
        except ValueError:
            psi_sup = np.nan

    return psi_sub, psi_sup


def parker_velocity_profile(T0, xi_array):
    """Compute the trans-sonic velocity profile for given coronal temperature.

    Args:
        T0: Coronal base temperature [K].
        xi_array: Array of dimensionless distances r/a.

    Returns:
        Array of velocities [km/s].
    """
    lam = compute_lambda(T0)
    xi_c = lam / 2.0  # Critical point

    # Sound speed
    c_s = np.sqrt(2 * k_B * T0 / M_H)  # cm/s

    # At critical point: psi = 1, F(xi_c)
    F_c = parker_equation_rhs(xi_c, lam)

    # Eigenvalue: psi_0 - ln(psi_0) = 1 - F_c + F(1) = 1 - F_c (since F(1)=0)
    Y_0 = 1.0 - F_c  # = 2*lambda - 3 - 4*ln(lambda/2)  (Parker Eq. 16)

    velocities = np.zeros_like(xi_array)

    for i, xi in enumerate(xi_array):
        F_xi = parker_equation_rhs(xi, lam)
        Y_xi = Y_0 + F_xi  # psi - ln(psi) = Y_xi

        if Y_xi < 1.0:
            velocities[i] = np.nan
            continue

        psi_sub, psi_sup = solve_psi_from_Y(Y_xi)

        if xi <= xi_c:
            psi = psi_sub
        else:
            psi = psi_sup

        if not np.isnan(psi):
            velocities[i] = c_s * np.sqrt(psi)  # v = c_s * sqrt(psi)
        else:
            velocities[i] = np.nan

    return velocities / 1e5  # Convert to km/s


# Reproduce Parker's Figure 1
xi_array = np.logspace(0, 2, 500)  # r/a from 1 to 100

T0_values = [0.5e6, 1.0e6, 1.5e6, 2.0e6, 2.5e6, 3.0e6, 4.0e6]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(T0_values)))

fig, ax = plt.subplots(figsize=(10, 7))

for T0, color in zip(T0_values, colors):
    v = parker_velocity_profile(T0, xi_array)
    label = f'T₀ = {T0/1e6:.1f}×10⁶ K'
    ax.plot(xi_array, v, color=color, linewidth=2, label=label)

    # Mark critical point
    lam = compute_lambda(T0)
    xi_c = lam / 2
    c_s = np.sqrt(2 * k_B * T0 / M_H) / 1e5  # km/s
    if 1 < xi_c < 100:
        ax.plot(xi_c, c_s, 'o', color=color, markersize=6, markeredgecolor='black')

ax.axvline(x=AU/a, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
ax.text(AU/a * 1.05, 850, '1 AU', fontsize=11, color='gray')

ax.set_xlabel('ξ = r/a (dimensionless distance / 무차원 거리)')
ax.set_ylabel('v(r) [km/s] (expansion velocity / 팽창 속도)')
ax.set_title("Parker's Figure 1: Isothermal Solar Wind Velocity\n"
             "Parker의 Fig. 1: 등온 태양풍 속도 (재현)")
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(1, 100)
ax.set_ylim(0, 950)
ax.grid(True, alpha=0.3)

# Annotate Biermann's required velocity
ax.axhline(y=500, color='red', linestyle='--', alpha=0.5)
ax.text(2, 520, "Biermann's required v = 500 km/s", fontsize=10, color='red')

plt.tight_layout()
plt.savefig('part2_parker_fig1.png', dpi=150, bbox_inches='tight')
plt.show()

print("● marks the critical (sonic) point for each temperature")
print("● 각 온도에서의 임계점(음속점)을 표시합니다")

## Part 3: Critical Point Analysis and Solution Classes
## 파트 3: 임계점 분석과 해의 분류

The Parker equation has multiple solution classes. Only the trans-sonic solution
(subsonic at base → supersonic at large r) is physically meaningful. Other solutions
represent accretion flows, breezes, or unphysical solutions.

Parker 방정식에는 여러 종류의 해가 있습니다. 물리적으로 유의미한 것은 trans-sonic 해
(기저부에서 아음속 → 먼 거리에서 초음속)뿐입니다.

In [ ]:
# Show all solution classes of the Parker equation
T0 = 2.0e6
lam = compute_lambda(T0)
xi_c = lam / 2.0
c_s = np.sqrt(2 * k_B * T0 / M_H)

xi_fine = np.linspace(1.01, 80, 1000)

# The Parker equation: psi - ln(psi) = Y(xi) where
# Y(xi) = Y_0 + 4*ln(xi) - 2*lambda*(1 - 1/xi)
# At the critical point xi_c = lambda/2, Y must = 1 for trans-sonic

F = lambda xi: 4.0 * np.log(xi) - 2.0 * lam * (1.0 - 1.0 / xi)
F_c = F(xi_c)

fig, ax = plt.subplots(figsize=(10, 8))

# Class I: Trans-sonic (solar wind) — THE physical solution
Y_0_transonic = 1.0 - F_c
v_trans = np.zeros_like(xi_fine)
for i, xi in enumerate(xi_fine):
    Y = Y_0_transonic + F(xi)
    if Y < 1.0:
        v_trans[i] = np.nan
        continue
    ps, pp = solve_psi_from_Y(Y)
    v_trans[i] = (c_s * np.sqrt(pp if xi > xi_c else ps)) / 1e5

ax.plot(xi_fine, v_trans, 'r-', linewidth=3, label='I: Trans-sonic (solar wind)\n    Trans-sonic (태양풍) ★')

# Class II: Trans-sonic (accretion) — supersonic at base, subsonic at large r
v_accr = np.zeros_like(xi_fine)
for i, xi in enumerate(xi_fine):
    Y = Y_0_transonic + F(xi)
    if Y < 1.0:
        v_accr[i] = np.nan
        continue
    ps, pp = solve_psi_from_Y(Y)
    v_accr[i] = (c_s * np.sqrt(ps if xi > xi_c else pp)) / 1e5

ax.plot(xi_fine, v_accr, 'b-', linewidth=2, label='II: Trans-sonic (accretion)\n     Trans-sonic (강착)')

# Class III & IV: Subsonic breezes and supersonic everywhere
for delta, style, label_class in [
    (0.5, '--', 'III: Subsonic breeze (아음속 바람)'),
    (1.5, '--', 'III: Subsonic breeze'),
    (-0.5, ':', 'IV: Supersonic everywhere (초음속)'),
    (-1.0, ':', 'IV: Supersonic everywhere'),
]:
    Y_0_shifted = Y_0_transonic + delta
    v_class = np.zeros_like(xi_fine)
    for i, xi in enumerate(xi_fine):
        Y = Y_0_shifted + F(xi)
        if Y < 1.0:
            v_class[i] = np.nan
            continue
        ps, pp = solve_psi_from_Y(Y)
        if delta > 0:  # Subsonic breeze
            v_class[i] = (c_s * np.sqrt(ps)) / 1e5
        else:  # Supersonic everywhere
            v_class[i] = (c_s * np.sqrt(pp)) / 1e5

    if 'III' in label_class:
        ax.plot(xi_fine, v_class, style, color='green', linewidth=1.5,
                label=label_class if delta == 0.5 else None, alpha=0.7)
    else:
        ax.plot(xi_fine, v_class, style, color='purple', linewidth=1.5,
                label=label_class if delta == -0.5 else None, alpha=0.7)

# Mark critical point
ax.plot(xi_c, c_s / 1e5, 'k*', markersize=15, zorder=10, label=f'Critical point\nξ_c = λ/2 = {xi_c:.1f}')

# Mark sound speed line
ax.axhline(y=c_s/1e5, color='gray', linestyle='-.', alpha=0.5, linewidth=1)
ax.text(2, c_s/1e5 + 10, f'Sound speed c_s = {c_s/1e5:.0f} km/s', fontsize=10, color='gray')

ax.set_xlabel('ξ = r/a')
ax.set_ylabel('v [km/s]')
ax.set_title(f'Solution Classes of Parker Equation (T₀ = {T0/1e6:.0f}×10⁶ K, λ = {lam:.1f})\n'
             f'Parker 방정식의 해 분류')
ax.legend(loc='upper left', fontsize=9)
ax.set_xlim(1, 80)
ax.set_ylim(0, 900)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('part3_solution_classes.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Only Class I (red, trans-sonic solar wind) satisfies:")
print(f"  1. Subsonic at r = a (coronal base)")
print(f"  2. Supersonic at large r")
print(f"  3. p → 0 as r → ∞")
print(f"\n물리적으로 유의미한 해는 Class I (빨간색, trans-sonic 태양풍)뿐입니다.")

## Part 4: The Parker Spiral — Interplanetary Magnetic Field
## 파트 4: Parker 나선 — 행성간 자기장

Parker's second great discovery: the combination of radial outflow ($v_m$) and solar rotation ($\omega$)
winds the magnetic field into an Archimedean spiral. The spiral angle is:

$$\tan\Psi = \frac{\omega(r-b)\sin\theta}{v_m}$$

Parker의 두 번째 대발견: 방사 유출($v_m$)과 태양 자전($\omega$)의 결합이 자기장을
Archimedean spiral로 감습니다.

In [ ]:
def parker_spiral(phi_range, v_m_kms, b_au=0.1):
    """Compute Parker spiral field line in the equatorial plane.

    The streamline equation (Parker Eq. 25):
    r/b - 1 - ln(r/b) = (v_m / b*omega) * (phi - phi_0)

    For large r: r ≈ -(v_m/omega) * phi (Archimedean spiral).

    Args:
        phi_range: Array of azimuthal angles [radians].
        v_m_kms: Asymptotic wind speed [km/s].
        b_au: Inner boundary radius [AU].

    Returns:
        Array of r values [AU].
    """
    v_m = v_m_kms * 1e5  # cm/s
    b = b_au * AU  # cm

    r_vals = np.zeros_like(phi_range)
    for i, phi in enumerate(phi_range):
        rhs = (v_m / (b * omega_sun)) * phi
        # Solve: r/b - 1 - ln(r/b) = rhs for r/b
        # For rhs > 0, r/b > 1
        try:
            x = brentq(lambda x: x - 1 - np.log(x) - rhs, 1.0001, 1e6)
            r_vals[i] = x * b / AU
        except ValueError:
            r_vals[i] = np.nan

    return r_vals


# Reproduce Parker's Figure 6 — spiral field lines
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Parker spiral for different wind speeds
ax1 = axes[0]

v_m_values = [400, 700, 1000]
colors_sp = ['#e74c3c', '#2ecc71', '#3498db']

for v_m, color in zip(v_m_values, colors_sp):
    for phi_0_deg in range(0, 360, 45):
        phi_range = np.linspace(0, 4 * np.pi, 500)
        r = parker_spiral(phi_range, v_m, b_au=0.05)

        phi_actual = phi_range + np.radians(phi_0_deg)
        x = r * np.cos(phi_actual)
        y = r * np.sin(phi_actual)

        mask = r < 6  # Limit to 6 AU
        label = f'v_m = {v_m} km/s' if phi_0_deg == 0 else None
        ax1.plot(x[mask], y[mask], color=color, linewidth=1.0, alpha=0.7, label=label)

# Sun and planets
ax1.plot(0, 0, 'yo', markersize=12, markeredgecolor='orange', zorder=10)
for r_planet, name in [(0.39, 'Mercury'), (0.72, 'Venus'), (1.0, 'Earth'), (1.52, 'Mars')]:
    circle = plt.Circle((0, 0), r_planet, fill=False, color='gray', linestyle=':', alpha=0.3)
    ax1.add_patch(circle)
    ax1.text(r_planet * 0.7, r_planet * 0.7, name, fontsize=8, color='gray')

ax1.set_xlim(-5, 5)
ax1.set_ylim(-5, 5)
ax1.set_aspect('equal')
ax1.set_xlabel('x [AU]')
ax1.set_ylabel('y [AU]')
ax1.set_title("Parker Spiral (Equatorial Plane)\nParker 나선 (적도면)")
ax1.legend(loc='lower left', fontsize=10)
ax1.grid(True, alpha=0.2)

# Right: Spiral angle vs distance
ax2 = axes[1]
r_au = np.linspace(0.1, 5, 200)
b_au = 0.05

for v_m, color in zip(v_m_values, colors_sp):
    tan_psi = omega_sun * (r_au * AU - b_au * AU) / (v_m * 1e5)
    psi = np.degrees(np.arctan(tan_psi))
    ax2.plot(r_au, psi, color=color, linewidth=2.5, label=f'v_m = {v_m} km/s')

ax2.axhline(y=45, color='gray', linestyle='--', alpha=0.5)
ax2.text(0.2, 47, '45° (B_ϕ = B_r)', fontsize=10, color='gray')
ax2.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5)
ax2.text(1.05, 5, 'Earth\n1 AU', fontsize=10, color='gray')

ax2.set_xlabel('Distance from Sun [AU] / 태양 거리')
ax2.set_ylabel('Spiral angle Ψ [°] / 나선각')
ax2.set_title('Parker Spiral Angle vs Distance\n나선각 vs 거리')
ax2.legend(fontsize=11)
ax2.set_xlim(0, 5)
ax2.set_ylim(0, 85)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('part4_parker_spiral.png', dpi=150, bbox_inches='tight')
plt.show()

# Print spiral angles at Earth
print("\nSpiral angle at 1 AU (Earth) / 1 AU(지구)에서의 나선각:")
for v_m in [300, 400, 500, 700, 1000]:
    tan_psi = omega_sun * (AU - b_au * AU) / (v_m * 1e5)
    psi = np.degrees(np.arctan(tan_psi))
    print(f"  v_m = {v_m:4d} km/s → Ψ = {psi:.1f}°")

print(f"\nObserved average at 1 AU: Ψ ≈ 45° (slow wind) / 관측 평균: ~45°")

## Part 5: Mass Loss and Comparison with Modern Observations
## 파트 5: 질량 손실과 현대 관측 비교

Parker's predictions vs modern in-situ measurements. The solar wind was confirmed
by Mariner 2 in 1962, and Parker's predictions were remarkably accurate.

Parker의 예측 vs 현대 현장 관측. 태양풍은 1962년 Mariner 2에 의해 확인되었으며,
Parker의 예측은 놀랍도록 정확했습니다.

In [ ]:
# Mass loss rate as a function of coronal temperature (Parker's Eq. 23)
T0_range = np.linspace(0.5e6, 4e6, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: v_0 and v(1AU) vs T0
ax1 = axes[0]
v_at_1AU = []
v_0_list = []

for T0 in T0_range:
    lam = compute_lambda(T0)
    c_s = np.sqrt(2 * k_B * T0 / M_H) / 1e5  # km/s
    xi_c = lam / 2

    # Eigenvalue condition: psi_0 - ln(psi_0) = 2*lambda - 3 - 4*ln(lambda/2)
    Y_eigenvalue = 2 * lam - 3 - 4 * np.log(lam / 2)
    if Y_eigenvalue >= 1:
        psi_0_sub, _ = solve_psi_from_Y(Y_eigenvalue)
        v0 = c_s * np.sqrt(psi_0_sub)
        v_0_list.append(v0)
    else:
        v_0_list.append(np.nan)

    # Velocity at 1 AU (xi = AU/a)
    xi_1AU = AU / a
    v_profile = parker_velocity_profile(T0, np.array([xi_1AU]))
    v_at_1AU.append(v_profile[0])

ax1.plot(T0_range/1e6, v_0_list, 'b-', linewidth=2, label='v₀ (coronal base / 기저부)')
ax1.plot(T0_range/1e6, v_at_1AU, 'r-', linewidth=2.5, label='v at 1 AU (지구 궤도)')
ax1.axhline(y=400, color='green', linestyle='--', alpha=0.5, label='Modern slow wind: 400 km/s')
ax1.axhline(y=700, color='green', linestyle=':', alpha=0.5, label='Modern fast wind: 700 km/s')
ax1.set_xlabel('T₀ [10⁶ K] / 코로나 온도')
ax1.set_ylabel('Velocity [km/s] / 속도')
ax1.set_title('Solar Wind Velocity vs Coronal Temperature\n태양풍 속도 vs 코로나 온도')
ax1.legend(fontsize=9)
ax1.set_ylim(0, 1000)
ax1.grid(True, alpha=0.3)

# Right: Mass loss rate
ax2 = axes[1]
mass_loss = []
for T0, v0 in zip(T0_range, v_0_list):
    if np.isnan(v0):
        mass_loss.append(np.nan)
    else:
        dMdt = 4 * np.pi * a**2 * N_0 * M_H * (v0 * 1e5)  # g/s
        mass_loss.append(dMdt)

ax2.semilogy(T0_range/1e6, mass_loss, 'r-', linewidth=2.5)
ax2.axhline(y=1e14, color='blue', linestyle='--', linewidth=2,
            label="Biermann's estimate: 10¹⁴ g/s")
ax2.axhline(y=2e12, color='green', linestyle='--', linewidth=2,
            label='Modern observed: ~2×10¹² g/s')
ax2.set_xlabel('T₀ [10⁶ K] / 코로나 온도')
ax2.set_ylabel('Mass loss rate [g/s] / 질량 손실률')
ax2.set_title('Solar Mass Loss Rate\n태양 질량 손실률')
ax2.legend(fontsize=10)
ax2.set_ylim(1e8, 1e16)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('part5_mass_loss.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary comparison table
print("=" * 70)
print("Parker (1958) vs Modern Observations / Parker vs 현대 관측")
print("=" * 70)
print(f"{'Parameter':<35} {'Parker':>15} {'Modern':>15}")
print("-" * 70)
print(f"{'Velocity at 1 AU [km/s]':<35} {'500-1000':>15} {'300-800':>15}")
print(f"{'  (for T₀ = 1.5-3×10⁶ K)':<35} {'':>15} {'':>15}")
print(f"{'Mass loss [g/s]':<35} {'~10¹⁴':>15} {'~2×10¹²':>15}")
print(f"{'Density at 1 AU [cm⁻³]':<35} {'~500':>15} {'3-10':>15}")
print(f"{'Spiral angle at 1 AU [°]':<35} {'20-45':>15} {'~45 (slow)':>15}")
print(f"{'Critical point [R_sun]':<35} {'5-20':>15} {'10-20':>15}")
print(f"{'Trans-sonic transition':<35} {'Predicted':>15} {'Confirmed':>15}")
print(f"{'Continuous outflow':<35} {'Predicted':>15} {'Confirmed':>15}")
print("-" * 70)
print("Parker's velocity predictions: EXCELLENT")
print("Parker's mass loss: overestimated (Biermann's N₀ was too high)")
print("Parker's spiral structure: CONFIRMED by Ness (1965)")
print("Parker의 속도 예측: 우수 / 질량 손실: 과대추정 / 나선 구조: Ness(1965) 확인")

## Summary Table / 요약 테이블

| Part | Topic / 주제 | Key Result / 핵심 결과 |
|------|-------------|----------------------|
| 1 | Hydrostatic failure / 정역학 실패 | p(∞) ≫ p_ISM → equilibrium impossible |
| 2 | Parker equation / Parker 방정식 | Trans-sonic solution reproduces Fig. 1 velocity curves |
| 3 | Solution classes / 해 분류 | Only one physically meaningful solution (trans-sonic wind) |
| 4 | Parker spiral / Parker 나선 | tan Ψ = ωr sinθ/v_m, ~45° at 1 AU for slow wind |
| 5 | Mass loss & comparison / 질량 손실 | Velocity prediction excellent; mass loss overestimated |

**Biermann (1951) → Parker (1958)**: Observation → Theory → Confirmation (Mariner 2, 1962)

**Next paper / 다음 논문**: Leighton, Noyes & Simon (1962) — "Velocity Fields in the Solar Atmosphere"
→ Discovery of 5-minute oscillations → birth of helioseismology
→ 5분 진동 발견 → 일진학의 탄생